# JetBot Tape Line Follower (Stabilized)
Smoother driving/detection. Fixes: single-tape lane-offset, error EMA, miss-grace, motor slew limit, fixed vertical bands + median aggregation, dt-scaled PID with anti-windup, morph close on mask.
Run cells top to bottom. Tune sliders while watching the live mask. Toggle **Enable Drive** when detection looks solid.

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import json
import time
from jetbot import Robot, Camera

In [ ]:
camera = Camera.instance(width=300, height=300, fps=10)
robot = Robot()

In [ ]:
# -- Stabilized all-in-one dashboard --

# Camera feeds
cam_img  = widgets.Image(format='jpeg', width=300, height=300)
mask_img = widgets.Image(format='jpeg', width=300, height=300)
info_lbl = widgets.HTML(value='<i>Waiting for frames...</i>')

# -- Detection mode --
mode = widgets.Dropdown(
    options=['Canny Edge', 'Adaptive Threshold'],
    value='Canny Edge', description='Mode')

# -- Canny sliders --
blur_k     = widgets.IntSlider(value=5,   min=1, max=21, step=2, description='Blur K')
canny_low  = widgets.IntSlider(value=50,  min=0, max=255, description='Canny Low')
canny_high = widgets.IntSlider(value=150, min=0, max=255, description='Canny High')
dilate_k   = widgets.IntSlider(value=3,   min=1, max=15, step=2, description='Dilate K')

# -- Adaptive threshold sliders --
block_size = widgets.IntSlider(value=11,  min=3,  max=51, step=2, description='Block Size')
c_offset   = widgets.IntSlider(value=2,   min=-20, max=20, description='C Offset')
invert     = widgets.ToggleButton(value=False, description='Invert',
                button_style='', layout=widgets.Layout(width='100px'))

# -- Mask cleanup --
close_k    = widgets.IntSlider(value=5,   min=1, max=15, step=2, description='Close K')

# -- Shared detection sliders --
min_run_w  = widgets.IntSlider(value=3,   min=1, max=30, description='Min Run W')
roi_pct    = widgets.FloatSlider(value=0.4,  min=0.1, max=1.0, step=0.05, description='ROI Bot %')
scan_rows  = widgets.IntSlider(value=12,  min=3, max=30, description='Scan Rows')
lane_w_px  = widgets.IntSlider(value=140, min=40, max=280, step=5, description='Lane Px')

# -- PID sliders --
kp         = widgets.FloatSlider(value=0.4,  min=0.0, max=2.0, step=0.01, description='Kp')
ki         = widgets.FloatSlider(value=0.0,  min=0.0, max=0.5, step=0.005, description='Ki')
kd         = widgets.FloatSlider(value=0.05, min=0.0, max=1.0, step=0.01, description='Kd')
err_ema    = widgets.FloatSlider(value=0.6,  min=0.0, max=0.95, step=0.05, description='Err EMA a')
integ_clip = widgets.FloatSlider(value=1.0,  min=0.1, max=5.0, step=0.1, description='I Clamp')

# -- Speed / drive sliders --
base_speed = widgets.FloatSlider(value=0.25, min=0.05, max=0.6, step=0.01, description='Base Speed')
curve_slow = widgets.FloatSlider(value=0.5,  min=0.0, max=1.0, step=0.05, description='Curve Slow')
lookahead  = widgets.FloatSlider(value=0.3,  min=0.0, max=1.0, step=0.05, description='Look-ahead')
min_wheel  = widgets.FloatSlider(value=0.0,  min=-0.5, max=0.15, step=0.01, description='Min Wheel')
slew_max   = widgets.FloatSlider(value=0.15, min=0.02, max=1.0, step=0.01, description='Slew/frame')
miss_grace = widgets.IntSlider(value=5,    min=0, max=20, description='Miss Grace')

# -- Drive toggle --
drive_enabled = widgets.ToggleButton(
    value=False, description='Enable Drive',
    button_style='danger', icon='car',
    layout=widgets.Layout(width='160px', height='40px'))

def on_drive_toggle(change):
    if change['new']:
        drive_enabled.button_style = 'success'
        drive_enabled.description = 'Driving!'
    else:
        drive_enabled.button_style = 'danger'
        drive_enabled.description = 'Enable Drive'
        robot.stop()
drive_enabled.observe(on_drive_toggle, names='value')

# -- Show/hide mode-specific sliders --
canny_box    = widgets.VBox([blur_k, canny_low, canny_high, dilate_k])
adaptive_box = widgets.VBox([block_size, c_offset, invert])
adaptive_box.layout.display = 'none'

def on_mode_change(change):
    if change['new'] == 'Canny Edge':
        canny_box.layout.display = ''
        adaptive_box.layout.display = 'none'
    else:
        canny_box.layout.display = 'none'
        adaptive_box.layout.display = ''
mode.observe(on_mode_change, names='value')

# -- Save / Load --
save_btn = widgets.Button(description='Save', button_style='info', icon='save')
load_btn = widgets.Button(description='Load', button_style='warning', icon='folder-open')
status_lbl = widgets.Label(value='')
CFG = 'line_follower_config_v2.json'

ALL_PARAMS = [('mode',mode),('blur_k',blur_k),('canny_low',canny_low),
              ('canny_high',canny_high),('dilate_k',dilate_k),
              ('block_size',block_size),('c_offset',c_offset),('invert',invert),
              ('close_k',close_k),('min_run_w',min_run_w),('roi_pct',roi_pct),
              ('scan_rows',scan_rows),('lane_w_px',lane_w_px),
              ('kp',kp),('ki',ki),('kd',kd),('err_ema',err_ema),('integ_clip',integ_clip),
              ('base_speed',base_speed),('curve_slow',curve_slow),
              ('lookahead',lookahead),('min_wheel',min_wheel),
              ('slew_max',slew_max),('miss_grace',miss_grace)]

def save_config(_):
    with open(CFG,'w') as f:
        json.dump({k: w.value for k,w in ALL_PARAMS}, f, indent=2)
    status_lbl.value = 'Saved!'
def load_config(_):
    try:
        with open(CFG) as f:
            cfg = json.load(f)
        for k, w in ALL_PARAMS:
            if k in cfg: w.value = cfg[k]
        status_lbl.value = 'Loaded!'
    except FileNotFoundError:
        status_lbl.value = 'No config file.'
save_btn.on_click(save_config)
load_btn.on_click(load_config)

# -- Layout --
left_col = widgets.VBox([
    widgets.HBox([cam_img, mask_img]),
    info_lbl
])

right_col = widgets.VBox([
    mode,
    widgets.HTML('<b>Detection</b>'),
    canny_box, adaptive_box, close_k, min_run_w, lane_w_px,
    widgets.HTML('<hr style="margin:4px 0"><b>PID / Steering</b>'),
    roi_pct, scan_rows, kp, ki, kd, err_ema, integ_clip,
    widgets.HTML('<hr style="margin:4px 0"><b>Speed / Drive</b>'),
    base_speed, curve_slow, lookahead, min_wheel, slew_max, miss_grace,
    drive_enabled,
    widgets.HBox([save_btn, load_btn, status_lbl]),
], layout=widgets.Layout(padding='0 0 0 12px'))

display(widgets.HBox([left_col, right_col]))

# -- Controller state --
_integral = 0.0
_prev_err_f = 0.0
_err_f = 0.0
_last_steer = 0.0
_last_lm = 0.0
_last_rm = 0.0
_miss_count = 0
_last_single_side = 0   # -1 = left tape seen alone, +1 = right tape seen alone
_last_t = None


def get_runs(row):
    # Contiguous nonzero runs in a 1D binary row.
    padded = np.concatenate(([0], row, [0]))
    diff = np.diff(padded.astype(np.int16))
    starts = np.where(diff > 0)[0]
    ends   = np.where(diff < 0)[0]
    return list(zip(starts, ends))


def find_lane_points(mask, rw):
    # Scan rows -> list of (row, li, ri, cx, conf).
    # conf 1.0 = both tapes visible, 0.5 = single tape inferred via lane_w_px offset.
    global _last_single_side
    rh = mask.shape[0]
    n = scan_rows.value
    row_indices = np.linspace(rh - 1, 0, n, dtype=int)
    mrw = min_run_w.value
    half_lane = lane_w_px.value // 2
    points = []
    for r in row_indices:
        runs = get_runs(mask[r])
        runs = [(s, e) for s, e in runs if (e - s) >= mrw]
        if len(runs) >= 2:
            left_tape = runs[0]
            right_tape = runs[-1]
            li = int(left_tape[1] - 1)
            ri = int(right_tape[0])
            cx = (li + ri) // 2
            points.append((int(r), li, ri, cx, 1.0))
        elif len(runs) == 1:
            s, e = runs[0]
            tape_center = (int(s) + int(e - 1)) // 2
            side = _last_single_side
            if side == 0:
                side = -1 if tape_center < rw // 2 else 1
            if side == -1:
                cx = tape_center + half_lane
            else:
                cx = tape_center - half_lane
            cx = int(np.clip(cx, 0, rw - 1))
            points.append((int(r), int(s), int(e - 1), cx, 0.5))
    return points


def update_single_side_hint(points, rw):
    # Refresh _last_single_side so future single-tape frames know which side's tape to expect.
    global _last_single_side
    if not points:
        return
    twos = [p for p in points if p[4] >= 1.0]
    if twos:
        bottom = twos[0]
        li, ri = bottom[1], bottom[2]
        if abs(li - rw // 2) < abs(ri - rw // 2):
            _last_single_side = -1
        else:
            _last_single_side = 1
    else:
        bottom = points[0]
        tape_center = (bottom[1] + bottom[2]) // 2
        _last_single_side = -1 if tape_center < rw // 2 else 1


def band_aggregate(points, h_roi, rw):
    # Fixed-band split: bottom 50% of ROI = near, top 50% = far. Median for robustness.
    if not points:
        return None, None, 0, 0
    near_list, far_list = [], []
    split_row = h_roi * 0.5
    for (r, li, ri, cx, _) in points:
        if r >= split_row:
            near_list.append(cx)
        else:
            far_list.append(cx)
    near_cx = float(np.median(near_list)) if near_list else None
    far_cx  = float(np.median(far_list))  if far_list  else None
    return near_cx, far_cx, len(near_list), len(far_list)


def process_frame(change):
    global _integral, _prev_err_f, _err_f, _last_steer
    global _last_lm, _last_rm, _miss_count, _last_t

    now = time.time()
    dt = 0.1 if _last_t is None else max(1e-3, min(0.5, now - _last_t))
    _last_t = now

    frame = change['new']
    h, w = frame.shape[:2]

    roi_top = int(h * (1.0 - roi_pct.value))
    roi = frame[roi_top:, :]
    h_roi = roi.shape[0]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    bk = blur_k.value | 1
    blurred = cv2.GaussianBlur(gray, (bk, bk), 0)

    if mode.value == 'Canny Edge':
        edges = cv2.Canny(blurred, canny_low.value, canny_high.value)
        dk = dilate_k.value | 1
        mask = cv2.dilate(edges, np.ones((dk, dk), np.uint8), iterations=2)
    else:
        bs = block_size.value | 1
        if bs < 3: bs = 3
        thresh_type = cv2.THRESH_BINARY_INV if not invert.value else cv2.THRESH_BINARY
        mask = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     thresh_type, bs, c_offset.value)
        kern = np.ones((3, 3), np.uint8)
        mask = cv2.erode(mask, kern, iterations=1)
        mask = cv2.dilate(mask, kern, iterations=2)

    # Morph close to glue broken tape runs
    ck = close_k.value | 1
    if ck >= 3:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,
                                np.ones((ck, ck), np.uint8))

    points = find_lane_points(mask, w)

    ov = frame.copy()
    cv2.line(ov, (0, roi_top), (w, roi_top), (0, 0, 255), 2)
    cv2.line(ov, (w // 2, roi_top), (w // 2, h), (128, 128, 128), 1)

    if points:
        update_single_side_hint(points, w)

        for r, lx, rx, cx, conf in points:
            yr = r + roi_top
            col_edge = (0, 0, 255) if conf >= 1.0 else (0, 180, 180)
            cv2.circle(ov, (lx, yr), 4, col_edge, -1)
            cv2.circle(ov, (rx, yr), 4, col_edge, -1)
            cv2.circle(ov, (cx, yr), 5, (0, 255, 0), -1)

        near_cx, far_cx, near_n, far_n = band_aggregate(points, h_roi, w)

        if near_cx is None and far_cx is None:
            _miss_count += 1
            _no_line(ov, h)
        else:
            if near_cx is None: near_cx = far_cx
            if far_cx  is None: far_cx  = near_cx

            near_err = (near_cx - w / 2) / (w / 2)
            far_err  = (far_cx  - w / 2) / (w / 2)

            la = lookahead.value * (1.0 if far_n >= 2 else 0.3)
            error = (1.0 - la) * near_err + la * far_err

            a = err_ema.value
            _err_f = a * _err_f + (1.0 - a) * error

            deriv = (_err_f - _prev_err_f) / dt
            leak = 0.97
            _integral = _integral * leak + _err_f * dt
            _integral = float(np.clip(_integral, -integ_clip.value, integ_clip.value))
            steer_raw = (kp.value * _err_f
                         + ki.value * _integral
                         + kd.value * deriv)
            steer = float(np.clip(steer_raw, -1.0, 1.0))
            if abs(steer_raw) > 1.0 and np.sign(steer_raw) == np.sign(_err_f):
                _integral -= _err_f * dt

            _prev_err_f = _err_f

            speed = base_speed.value * (1.0 - curve_slow.value * abs(steer))

            lm = speed + steer
            rm = speed - steer
            mw = min_wheel.value
            lm = float(np.clip(lm, mw, 1.0))
            rm = float(np.clip(rm, mw, 1.0))

            sm = slew_max.value
            lm = float(np.clip(lm, _last_lm - sm, _last_lm + sm))
            rm = float(np.clip(rm, _last_rm - sm, _last_rm + sm))

            if drive_enabled.value:
                robot.set_motors(lm, rm)
                _last_lm, _last_rm = lm, rm
            else:
                _last_lm, _last_rm = 0.0, 0.0

            _last_steer = steer
            _miss_count = 0

            near_y = (points[0][0] if points else h_roi - 1) + roi_top
            far_y  = (points[-1][0] if points else 0) + roi_top
            cv2.circle(ov, (int(near_cx), near_y), 10, (0, 255, 255), -1)
            cv2.circle(ov, (int(far_cx), far_y), 8, (255, 255, 0), 2)
            blend_x = int((1.0 - la) * near_cx + la * far_cx)
            cv2.line(ov, (w // 2, near_y), (blend_x, near_y), (255, 0, 255), 2)

            conf_avg = np.mean([p[4] for p in points])
            info_lbl.value = (f'<b style="color:green">TRACKING</b> &nbsp; '
                             f'err:{error:+.2f} ef:{_err_f:+.2f} &nbsp; '
                             f'steer:{steer:+.2f} &nbsp; spd:{speed:.2f} &nbsp; '
                             f'L:{lm:.2f} R:{rm:.2f} &nbsp; '
                             f'conf:{conf_avg:.2f} near:{near_n} far:{far_n} dt:{dt*1000:.0f}ms')
    else:
        _miss_count += 1
        _no_line(ov, h)

    _, j1 = cv2.imencode('.jpg', ov)
    cam_img.value = j1.tobytes()
    mask_full = np.zeros((h, w), np.uint8)
    mask_full[roi_top:, :] = mask
    _, j2 = cv2.imencode('.jpg', cv2.cvtColor(mask_full, cv2.COLOR_GRAY2BGR))
    mask_img.value = j2.tobytes()


def _no_line(ov, h):
    # Grace period: hold last steer + decay speed while _miss_count <= miss_grace.
    # After that, full stop + PID reset.
    global _integral, _prev_err_f, _err_f, _last_lm, _last_rm
    grace = miss_grace.value
    if _miss_count <= grace and drive_enabled.value:
        decay = 0.6
        lm = _last_lm * decay
        rm = _last_rm * decay
        sm = slew_max.value
        lm = float(np.clip(lm, _last_lm - sm, _last_lm + sm))
        rm = float(np.clip(rm, _last_rm - sm, _last_rm + sm))
        robot.set_motors(lm, rm)
        _last_lm, _last_rm = lm, rm
        cv2.putText(ov, f'HOLD {_miss_count}/{grace}', (10, h // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 180, 255), 2)
        info_lbl.value = (f'<b style="color:orange">HOLDING</b> '
                         f'{_miss_count}/{grace} - L:{lm:.2f} R:{rm:.2f}')
    else:
        _integral = 0.0
        _prev_err_f = 0.0
        _err_f = 0.0
        _last_lm = 0.0
        _last_rm = 0.0
        if drive_enabled.value:
            robot.stop()
        cv2.putText(ov, 'LINE LOST', (10, h // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        info_lbl.value = '<b style="color:red">LINE LOST</b> - motors stopped'


camera.observe(process_frame, names='value')
print('Stabilized dashboard active. Adjust sliders live, toggle Enable Drive when ready.')

In [ ]:
# -- STOP -- run this cell to halt everything
camera.unobserve_all()
robot.stop()
drive_enabled.value = False
print('Stopped.')